# 08 — Model 4: Deep 1D CNN (fourth rung of the complexity ladder)

Fourth rung of the Sleep-EDF complexity ladder (parameter count = the complexity axis) and the **largest,
deepest CNN**. Model 2 was shallow `[16, 32]` (~8.2K params), Model 3 medium `[32, 64, 64]` (~93K); Model 4
is the **deep CNN** `[64, 128, 128, 256]` from `harness/models/cnn.py` (~864K params) — four blocks — trained
on the **raw** single-channel EEG (no band features, no bandpass filter — only the loader's per-epoch
z-normalisation).

**Controlled ladder.** Only the variant changes from Model 3. Data split, test set, kernel size, stopping
rule, seeds, device, batch size, optimizer and the multi-seed helper are all held identical, so any
difference in results (including faithfulness later) is attributable to complexity, not to a confounded
training change.

**Ladder context.** Model 4's receptive field is **2260 ms**, which sits **above** the AASM stage-defining
event scale (~500–1500 ms: spindles, K-complexes, slow waves). Across the CNN rungs the receptive field
sweeps through that scale: Model 2's 460 ms sat at the lower edge (a minimal event), Model 3's 1060 ms fell
inside it (one full event), and Model 4's 2260 ms spans **multiple events plus surrounding context**.
Recorded here so the complexity axis (parameters) and the mechanistic reading (receptive field vs event
scale) travel with the model.

**This notebook is TRAINING only.** The XAI + CMI faithfulness analysis is a separate notebook, after all
CNNs are trained. No coefficient/known-answer sanity check (no CNN analogue) — faithfulness belongs there.

Sections: **1** setup · **2** data · **3** model + parameter count · **4** multi-seed training (you run it) ·
**5** learning verification + ladder comparison (you run it after training).

> **You run the training.** Section 4's LAUNCH cell times seed 0, prints a total-time estimate, then trains
> all 5 seeds unattended (MPS), saving each seed as it finishes. This is the largest CNN — **check seed 0's
> estimate before letting the run continue** (MAX_EPOCHS is in §1, one place, if you need to lower it).

## 1. Setup, imports & config

In [ ]:
import sys
from pathlib import Path
# Locate repo root robustly: walk up to the dir containing sleep_edf/ (depth-independent).
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "sleep_edf").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             confusion_matrix, precision_recall_fscore_support)

from sleep_edf.loader import load_sleep_edf                 # full test set (never subsampled)
from sleep_edf.validation import train_val_split            # fixed subject-level 17,742 / 2,258 split
import sleep_edf.config as cfg
from sleep_edf.training import run_all_seeds                # shared timed/progress/save driver
# Shared CNN architecture + training recipe (identical across the whole ladder):
from harness.models.cnn import (build_cnn, train_cnn, set_seed, torch_predict_proba,
                                 count_parameters, CNN_VARIANTS)

CLASS_NAMES = ["W", "N1", "N2", "N3", "REM"]
SEEDS       = [0, 1, 2, 3, 4]        # 5 seeds — fixed up front (see §4), no single-seed preview
VARIANT     = "deep"                 # ladder rung: CNN_VARIANTS["deep"] = [64, 128, 128, 256]
MODEL_NAME  = "model4_deep_cnn"
# Previous rungs (in ladder order), read from saved JSON for the comparison cell (§5).
PREV_MODEL_NAMES = ["model2_shallow_cnn", "model3_medium_cnn"]

# ── Ladder-wide early-stopping protocol (fixed across Models 2-5; see DECISIONS_LOG) ──
# Monitor validation BALANCED ACCURACY, not loss: val loss is dominated by W+N2 (~69%),
# so a model can lower loss while N1/N3 recall degrades. These are passed to train_cnn
# (they are NOT harness defaults). MAX_EPOCHS is the one knob to revise after seed 0's timing.
STOP_MONITOR   = "val_balanced_accuracy"
STOP_MODE      = "max"
STOP_PATIENCE  = 10
STOP_MIN_DELTA = 0.002
MAX_EPOCHS     = 100          # <<< ceiling — change HERE (one place) after seeing seed 0's timing

# ── Training device (ladder-wide for Models 2-5) ──────────────────────────────
# MPS (Apple-Silicon GPU) is ~100x faster than CPU for this CNN (verified: ~2 ms/batch vs
# ~235 ms/batch), and is deterministic here (bit-identical across repeats of a seed). Falls
# back to CPU on non-Apple machines. Passed to train_cnn / torch_predict_proba; not a harness
# default. Numerics differ slightly from CPU (float32 across backends) — see DECISIONS_LOG.
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

OUT_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "metrics"
FIG_DIR  = PROJECT_ROOT / "sleep_edf" / "results" / "figures"
CKPT_DIR = PROJECT_ROOT / "sleep_edf" / "results" / "checkpoints"   # trained weights for the XAI notebook
for d in (OUT_DIR, FIG_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)
np.set_printoptions(precision=3, suppress=True)
print("variant:", VARIANT, CNN_VARIANTS[VARIANT], "| kernel (from config):", cfg.CNN_KERNEL_SIZE,
      "| seeds:", SEEDS)
print(f"stopping rule: monitor={STOP_MONITOR} mode={STOP_MODE} patience={STOP_PATIENCE} "
      f"min_delta={STOP_MIN_DELTA} max_epochs={MAX_EPOCHS}")
print(f"training device: {DEVICE}"
      + ("  (Apple-Silicon GPU)" if DEVICE == "mps" else "  (MPS unavailable -> CPU fallback)"))
print("results ->", OUT_DIR, "| checkpoints ->", CKPT_DIR)

## 2. Data loading

`train_val_split()` (from `sleep_edf/validation.py`) returns the **fixed subject-level split** of the
identical 20,000-epoch subsample every ladder model uses: **17,742 train / 2,258 val**, with 6 whole
subjects held out for validation (leakage-free early stopping — see DECISIONS_LOG). It is *loaded, not
constructed here* — no resampling, no rebalancing. `load_sleep_edf("test")` is the **full** 40,145-epoch
test set, never subsampled. Raw signal only (per-epoch z-norm from the loader; no bandpass).

In [ ]:
# Fixed subject-level split of the 20K subsample (train 17,742 / val 2,258) + full test set.
X_tr, y_tr, X_val, y_val = train_val_split()          # loads the frozen split; asserts its integrity
X_test, y_test           = load_sleep_edf("test")     # full 40,145, never subsampled

# Visible confirmation the right split loaded (per-class counts, train vs val).
print(f"train {X_tr.shape}  |  val {X_val.shape}  |  test {X_test.shape}")
print(f"\n{'stage':<6}{'train':>8}{'val':>8}{'test':>8}")
for c, cn in enumerate(CLASS_NAMES):
    print(f"{cn:<6}{int((y_tr==c).sum()):>8}{int((y_val==c).sum()):>8}{int((y_test==c).sum()):>8}")
print(f"{'TOTAL':<6}{len(y_tr):>8}{len(y_val):>8}{len(y_test):>8}")
# Guard: the fixed split must be exactly 17,742 / 2,258 (the committed ladder-wide numbers).
assert len(y_tr) == 17742 and len(y_val) == 2258, (len(y_tr), len(y_val))

# Reference points for the learning check (§5), from the TEST distribution.
test_counts = np.bincount(y_test, minlength=cfg.N_CLASSES)
floor = test_counts.max() / len(y_test); chance_balanced = 1.0 / cfg.N_CLASSES
print(f"\nmajority-class floor (always predict {CLASS_NAMES[int(test_counts.argmax())]}): {floor:.4f}"
      f"  | balanced-acc chance: {chance_balanced:.3f}")
print("N3 and N1 are the load-bearing minorities — keep them explicit in §5.")

## 3. Model definition & parameter count

The deep variant `[64, 128, 128, 256]` is built from the **shared** `harness.models.cnn.build_cnn` — no
architecture code is copied into `sleep_edf/`. Its first-layer **kernel_size comes from `sleep_edf/config.py`**
(`CNN_KERNEL_SIZE = 15` = 150 ms at 100 Hz, matched to Sleep-EDF's ~15-sample autocorrelation), **not** the
harness default of 7 — the *same* kernel as Models 2 and 3, so only depth/width change across the rung. The
cell below **hard-asserts the config kernel actually reached the conv layer** — it raises rather than silently
training at kernel 7 (which would place this rung at the wrong point on the ladder). It also records the two
axes reported side by side: **parameter count** (complexity, expected ~864K) and **receptive field**
(mechanistic reading, expected 226 samples / 2260 ms).

In [ ]:
# Build the deep CNN with Sleep-EDF shapes; kernel_size from config (NOT the harness default 7).
probe = build_cnn(VARIANT, n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS,
                  kernel_size=cfg.CNN_KERNEL_SIZE)

# HARD CHECK: the config kernel reached the first conv. Stop loudly if not.
k_used = probe.features[0].kernel_size[0]
if k_used != cfg.CNN_KERNEL_SIZE:
    raise RuntimeError(f"kernel mismatch: config says {cfg.CNN_KERNEL_SIZE} but conv1 has {k_used} "
                       "— refusing to train at the wrong ladder point.")

# Parameter count, split conv trunk vs classifier head.
trunk = sum(p.numel() for n, p in probe.named_parameters() if p.requires_grad and not n.startswith("head"))
head  = sum(p.numel() for n, p in probe.named_parameters() if p.requires_grad and n.startswith("head"))
n_params = count_parameters(probe)
assert trunk + head == n_params

# Receptive field (conv 'same' padding + one MaxPool(2) per block) and length bookkeeping.
def receptive_field(channels, k):
    rf, jump = 1, 1
    for _ in channels:
        rf += (k - 1) * jump          # conv (same padding preserves length)
        rf += (2 - 1) * jump; jump *= 2   # MaxPool(2) halves length
    return rf
RF = receptive_field(CNN_VARIANTS[VARIANT], cfg.CNN_KERNEL_SIZE)
down = 2 ** len(CNN_VARIANTS[VARIANT])
with torch.no_grad():
    seq_at_head = probe.features(torch.zeros(1, cfg.IN_CHANNELS, cfg.INPUT_LENGTH)).shape[-1]

print(f"variant={VARIANT} {CNN_VARIANTS[VARIANT]} | kernel={cfg.CNN_KERNEL_SIZE} "
      f"(padding=k//2={cfg.CNN_KERNEL_SIZE // 2}, odd -> length preserved by each conv)")
print(f"trainable params: {n_params:,}   (conv trunk {trunk:,}  +  head {head:,})")
print(f"receptive field : {RF} samples = {RF * 1000 // cfg.SAMPLING_RATE} ms at {cfg.SAMPLING_RATE} Hz")
print(f"downsampling    : 1/{down}   sequence length reaching the head: {seq_at_head}")

# Flags against the recorded Model-4 ladder figures (~864K params, RF 226 samples / 2260 ms,
# downsampling 1/16, sequence length at head 187).
if not (850000 <= n_params <= 875000):
    print(f"  !! FLAG: parameter count {n_params:,} is outside the expected ~864K — check before training.")
if RF != 226:
    print(f"  !! FLAG: receptive field {RF} != expected 226 samples — check before training.")
if down != 16:
    print(f"  !! FLAG: downsampling 1/{down} != expected 1/16 — check before training.")
if seq_at_head != 187:
    print(f"  !! FLAG: sequence length at head {seq_at_head} != expected 187 — check before training.")
del probe  # a fresh model is built per seed in §4

## 4. Multi-seed training (5 seeds — you run the LAUNCH cell)

**5 seeds, decided up front** (no single-seed preview): the CNN is trained from 5 random initialisations to
report mean ± spread. The shared driver `sleep_edf.training.run_all_seeds` times seed 0, prints a total-time
estimate, then trains the rest **unattended**, saving each seed to disk as it finishes (`resume=True` skips
seeds already saved).

**Training recipe.** Each seed builds a fresh deep CNN and trains it with the **shared** harness recipe
`train_cnn` — the *identical* recipe used by every CNN/transformer rung (Adam lr 1e-3, weight-decay 1e-4,
batch 16, class-weighted cross-entropy). The per-seed function passes the **ladder-wide stopping rule** set
in §1 straight to `train_cnn` (`monitor='val_balanced_accuracy'`, `mode='max'`, `patience=10`,
`min_delta=0.002`, `max_epochs=100`) — early stopping tracks balanced accuracy on the fixed val split and
restores the best-by-balanced-accuracy checkpoint. The driver stays generic; nothing about the stopping rule
lives in the driver.

**Per seed we record** `stopped_epoch` and `stop_reason` (`patience` vs `max_epochs`) from `train_cnn`, so §5
can show whether any seed hit the ceiling (i.e. was truncated while still improving). Each seed's trained
weights are also saved to `results/checkpoints/` — a CNN can't be reconstructed from summary numbers the way
the logistic model could, so the later XAI notebook loads them instead of retraining.

**Device.** Training runs on `DEVICE` (set in §1: MPS on Apple Silicon, ~100x faster than CPU for this CNN;
CPU fallback elsewhere) — passed to `train_cnn` and `torch_predict_proba`. MPS is deterministic here
(bit-identical across repeats of a seed), so the 5-seed spread is real seed variance; numerics differ
slightly from CPU (float32 across backends), so these results are not bit-comparable to a CPU run.

**Progress display note.** `train_cnn` exposes no per-epoch callback, so the driver's optional inner epoch
bar isn't driven — the **per-seed** outer bar and the seed-0 time estimate still show. (Not adding a hook: no
harness changes in this task.)

In [ ]:
def _predict_in_batches(pp, X, bs=512):
    """Batched inference: the harness predict_proba runs the whole array in one forward
    (fine for ECG200's ~100 samples); chunk it for Sleep-EDF's 40k test to avoid a huge
    intermediate activation. Not a training loop and does not touch the harness."""
    return np.concatenate([pp(X[i:i + bs]) for i in range(0, len(X), bs)], axis=0)

def train_one_seed(seed, tick=None):
    # `tick` is accepted for the driver contract but train_cnn has no per-epoch hook, so it
    # is never called (only the outer per-seed bar shows). Not modifying harness to add one.
    set_seed(seed)
    model = build_cnn(VARIANT, n_classes=cfg.N_CLASSES, in_channels=cfg.IN_CHANNELS,
                      kernel_size=cfg.CNN_KERNEL_SIZE)
    assert model.features[0].kernel_size[0] == cfg.CNN_KERNEL_SIZE           # guard every seed
    info = train_cnn(model, X_tr, y_tr, X_val, y_val, seed=seed, device=DEVICE,   # shared recipe +
                     monitor=STOP_MONITOR, mode=STOP_MODE, patience=STOP_PATIENCE,  # ladder-wide
                     min_delta=STOP_MIN_DELTA, max_epochs=MAX_EPOCHS)               # stopping rule

    ckpt = CKPT_DIR / f"{MODEL_NAME}_seed{seed}.pt"                          # weights for the XAI notebook
    torch.save(model.state_dict(), ckpt)                                     # state_dict is device-agnostic

    pp = torch_predict_proba(model, device=DEVICE)   # model is on DEVICE -> predict there, returns CPU numpy
    yp = _predict_in_batches(pp, X_test).argmax(1)
    pr, rc, f1c, sup = precision_recall_fscore_support(
        y_test, yp, labels=list(range(cfg.N_CLASSES)), zero_division=0)
    return {
        "accuracy":          float(accuracy_score(y_test, yp)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, yp)),
        "macro_f1":          float(f1_score(y_test, yp, average="macro")),
        "n_params":          int(count_parameters(model)),
        "variant":           VARIANT,
        "kernel_size":       int(cfg.CNN_KERNEL_SIZE),
        "best_epoch":        int(info["best_epoch"]),        # best val balanced-accuracy epoch
        "stopped_epoch":     int(info["stopped_epoch"]),     # last epoch actually run
        "stop_reason":       info["stop_reason"],            # 'patience' | 'max_epochs'
        "best_val_bal_acc":  float(info["best_metric"]),     # monitored metric at the restored ckpt
        "checkpoint":        str(ckpt.relative_to(PROJECT_ROOT)),
        "confusion":         confusion_matrix(y_test, yp, labels=list(range(cfg.N_CLASSES))).tolist(),
        "per_class":         {CLASS_NAMES[c]: {"precision": float(pr[c]), "recall": float(rc[c]),
                                               "f1": float(f1c[c]), "support": int(sup[c])}
                              for c in range(cfg.N_CLASSES)},
    }

print("train_one_seed defined. Run the LAUNCH cell below to train all 5 seeds (unattended).")

### 4 ▶ LAUNCH TRAINING — run this one cell (unattended, MPS)

**This is the cell you run to train.** It times seed 0, prints an estimated total time for all 5 seeds
(clearly visible **before** the long run continues), then trains the rest automatically, saving each seed to
`sleep_edf/results/metrics/model4_deep_cnn_seed{seed}.json` and its weights to
`sleep_edf/results/checkpoints/`. `resume=True` skips seeds already on disk — delete those files (or set
`resume=False`) to re-train fresh.

After seed 0, check the printed timing: if a full `max_epochs=100` run would be too long, stop, lower
`MAX_EPOCHS` in §1, and re-launch.

In [ ]:
# ▶▶▶ LAUNCH: trains all 5 seeds unattended (times seed 0, estimates, then continues automatically) ◀◀◀
aggregate = run_all_seeds(
    train_one_seed, SEEDS, OUT_DIR, MODEL_NAME,
    summary_keys=["balanced_accuracy", "accuracy", "macro_f1", "stopped_epoch"],
    resume=True,        # delete model4_deep_cnn_seed*.json (or resume=False) to force a fresh re-train
)

## 5. Learning verification (run after training)

Reads the saved aggregate (works after the LAUNCH cell or a kernel restart) and reports results **across the
5 seeds** (mean ± spread), never a single seed:

- **Balanced accuracy** against *both* references — the ~34% majority-class floor (always-W) and the 0.20
  five-class chance line.
- **Overall accuracy.**
- **Per-class recall and F1**, with **N1 and N3 explicit** (the load-bearing minorities).
- **Confusion matrix** (counts + row-normalised), same format as Model 1.
- **stopped_epoch / stop_reason across seeds** — if seeds routinely stop by `max_epochs`, they were
  truncated while still improving (raise `MAX_EPOCHS` in §1).

In [ ]:
import json
agg = json.load(open(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))
per_seed = agg["per_seed"]; s = agg["summary"]

n_params = per_seed[str(SEEDS[0])]["n_params"]
print(f"Model 4 — deep CNN | params {n_params:,} | kernel {per_seed[str(SEEDS[0])]['kernel_size']} "
      f"| {len(SEEDS)} seeds | trained on 17,742 (fixed 20K minus 6 val subjects)\n")

print(f"accuracy      : {s['accuracy']['mean']:.4f} ± {s['accuracy']['std']:.4f}   "
      f"(majority floor {floor:.4f}; margin {s['accuracy']['mean'] - floor:+.4f})")
print(f"balanced acc  : {s['balanced_accuracy']['mean']:.4f} ± {s['balanced_accuracy']['std']:.4f}   "
      f"(chance {chance_balanced:.3f}; margin {s['balanced_accuracy']['mean'] - chance_balanced:+.4f})")
print(f"macro-F1      : {s['macro_f1']['mean']:.4f} ± {s['macro_f1']['std']:.4f}")

print(f"\n{'stage':<6}{'recall':>9}{'f1':>9}{'precision':>11}{'test_support':>14}   (minority?)")
print('-' * 60)
for cn in CLASS_NAMES:
    R = np.mean([per_seed[str(sd)]['per_class'][cn]['recall']    for sd in SEEDS])
    F = np.mean([per_seed[str(sd)]['per_class'][cn]['f1']        for sd in SEEDS])
    P = np.mean([per_seed[str(sd)]['per_class'][cn]['precision'] for sd in SEEDS])
    sup = per_seed[str(SEEDS[0])]['per_class'][cn]['support']
    tag = "  <-- minority" if cn in ("N3", "N1") else ""
    print(f"{cn:<6}{R:>9.3f}{F:>9.3f}{P:>11.3f}{sup:>14}{tag}")

# stopped_epoch / stop_reason across seeds — is early stopping halting, or hitting the ceiling?
print(f"\n{'seed':<6}{'stopped_epoch':>15}{'stop_reason':>14}{'best_epoch':>12}{'best_val_bal':>14}")
print('-' * 61)
for sd in SEEDS:
    r = per_seed[str(sd)]
    print(f"{sd:<6}{r['stopped_epoch']:>15}{r['stop_reason']:>14}{r['best_epoch']:>12}{r['best_val_bal_acc']:>14.4f}")
n_ceiling = sum(per_seed[str(sd)]['stop_reason'] == 'max_epochs' for sd in SEEDS)
print(f"\nseeds stopped by max_epochs (ceiling): {n_ceiling}/{len(SEEDS)}"
      + ("  -> truncated while improving; raise MAX_EPOCHS in §1." if n_ceiling else "  -> all halted by patience (good)."))

In [ ]:
# Confusion matrix, summed over seeds — counts + row-normalised (recall per true stage). Model-1 format.
cm = np.sum([np.array(per_seed[str(sd)]['confusion']) for sd in SEEDS], axis=0)
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, M, title, fmt in [(ax[0], cm, "Confusion (counts, summed over seeds)", "d"),
                         (ax[1], cmn, "Row-normalised (recall per true stage)", ".2f")]:
    a.imshow(M, cmap="Blues", vmin=0, vmax=(M.max() if fmt == "d" else 1))
    for i in range(5):
        for j in range(5):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center", fontsize=8,
                   color="white" if M[i, j] > (M.max() * 0.5 if fmt == "d" else 0.5) else "black")
    a.set_xticks(range(5)); a.set_xticklabels(CLASS_NAMES)
    a.set_yticks(range(5)); a.set_yticklabels(CLASS_NAMES)
    a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(title)
fig.suptitle("Model 4 (deep CNN) — confusion across 5 seeds", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "sleep_edf_08_model4_confusion.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved:", (FIG_DIR / "sleep_edf_08_model4_confusion.png").relative_to(PROJECT_ROOT))

### Ladder so far — Models 2, 3, 4 (all three CNN rungs)

The three CNN rungs side by side, so the ladder is visible as it builds. Models 2 and 3 are read from their
saved aggregates (`model2_shallow_cnn_aggregate.json`, `model3_medium_cnn_aggregate.json`) — nothing
hardcoded; each receptive field is recomputed from the `variant`/`kernel_size` recorded in that file. As
parameters (8.2K → 93K → 864K) and receptive field (460 → 1060 → 2260 ms) climb, watch balanced accuracy,
macro-F1 and the minority recalls (N1, N3): this is the complexity axis the faithfulness study will later
read against, and the per-step deltas show where accuracy gains are still coming from.

In [ ]:
# Compare all three CNN rungs. Earlier rungs' numbers come from their saved JSON (not hardcoded).
def _rf_from_variant(variant, k):
    rf, jump = 1, 1
    for _ in CNN_VARIANTS[variant]:
        rf += (k - 1) * jump; rf += (2 - 1) * jump; jump *= 2
    return rf

def _rung(agg_path):
    a = json.load(open(agg_path)); ps = a["per_seed"]; sm = a["summary"]; sds = [str(s) for s in a["seeds"]]
    p0 = ps[sds[0]]
    return {
        "name": a["model"], "params": p0["n_params"],
        "rf": _rf_from_variant(p0["variant"], p0["kernel_size"]),
        "bal": sm["balanced_accuracy"]["mean"], "bal_sd": sm["balanced_accuracy"]["std"],
        "mf1": sm["macro_f1"]["mean"],
        "n1_rec": float(np.mean([ps[s]["per_class"]["N1"]["recall"] for s in sds])),
        "n3_rec": float(np.mean([ps[s]["per_class"]["N3"]["recall"] for s in sds])),
    }

# ladder order: previous rungs (from saved JSON) then this model
rungs = [_rung(OUT_DIR / f"{m}_aggregate.json") for m in PREV_MODEL_NAMES]
rungs.append(_rung(OUT_DIR / f"{MODEL_NAME}_aggregate.json"))

print(f"{'rung':<22}{'params':>9}{'RF (ms)':>10}{'bal acc':>12}{'macro-F1':>10}{'N1 rec':>9}{'N3 rec':>9}")
print('-' * 81)
for r in rungs:
    print(f"{r['name']:<22}{r['params']:>9,}{r['rf']*1000//cfg.SAMPLING_RATE:>9}{'ms'}"
          f"{r['bal']:>9.4f}±{r['bal_sd']:.3f}{r['mf1']:>10.4f}{r['n1_rec']:>9.3f}{r['n3_rec']:>9.3f}")
print('-' * 81)
# per-step deltas (rung N vs rung N-1)
for a, b in zip(rungs[:-1], rungs[1:]):
    label = f"Δ ({b['name'].split('_')[0]} - {a['name'].split('_')[0]})"
    print(f"{label:<22}{b['params']-a['params']:>+9,}"
          f"{(b['rf']-a['rf'])*1000//cfg.SAMPLING_RATE:>+8}ms"
          f"{b['bal']-a['bal']:>+10.4f}{b['mf1']-a['mf1']:>+10.4f}"
          f"{b['n1_rec']-a['n1_rec']:>+9.3f}{b['n3_rec']-a['n3_rec']:>+9.3f}")